Environment

In [1]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}

In [2]:
import sys
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [3]:
import os
import json
import pycomet
import numpy as np
import tensorflow as tf

from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils
from pycomet.networks import NodePartitionFunctions, EdgeBackward, LinearMEGraphNet

Plotting

In [4]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x,
    y,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [5]:
with open("./../physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [6]:
sol = h5todict(h5file='./../physics/solution.hdf5')

In [7]:
na, nm = [sol[f"X_{s}"] * sol['n'] for s in sp_th.keys()]

In [8]:
# Compute source term
t, T = [sol[k] for k in ('t', 'T')]
n = np.hstack([na, nm])
omega = kin.function(t=t, y=[n, T])
dnmdt = omega[:,1:-1] / const.UNA
dnmdt = dnmdt.numpy().squeeze()

Construct graph

In [9]:
# Adjacency matrix (forward)
kf = 0.0
for reaction in kin.reactions:
    kf = kf + reaction.fwd.k.numpy().squeeze()[0]
k_idx = np.where(kf > 0)
# Dimension
nb_pts, nb_nodes = nm.shape

In [10]:
# Nodes
node_nm = nm[0].reshape(1,-1,1)
node_nm = np.tile(node_nm, (nb_pts,1,1))
node_Tt = np.tile(T.reshape(-1,1,1), (1,nb_nodes,1))
node_Ti = np.tile(T.reshape(-1,1,1), (1,nb_nodes,1))

In [11]:
# Edges
edge_kf = kf[k_idx].reshape(1,-1,1)
edge_kf = np.tile(edge_kf, (nb_pts,1,1))
edge_na = na[0].reshape(1,1,1)
edge_na = np.tile(edge_na, (nb_pts,1,1))

In [12]:
# Graph
graph = [node_nm, node_Tt, node_Ti, edge_kf, edge_na]
graph = tf.nest.map_structure(tf.convert_to_tensor, graph)
# Target
target = tf.convert_to_tensor(nm[...,None]) / const.UNA

Load linear master equation graph model

In [13]:
lays_args = {

    "node_pf": {
        "lay_id": "NodePartitionFunctions",
        "kwargs": {
            "node_index_temp": {
                "int": 2,
                "chem": 1,
                "trans": 1
            },
            "specie_thermos": {
                "filename": "./data/sp_th.json",
                "thermo_process": 0,
                "eos_model": "PerfectGas"
            }
        }
    },

    "edge_bwd": {
        "lay_id": "EdgeBackward",
        "kwargs": {
            "node_index_pf": [3],
            "static_fwd_edge_index": "./data/fwd_edge_index.txt"
        }
    },

    "lin_me_gnet": {
        "lay_id": "LinearMEGraphNet",
        "kwargs": {
            "node_attr_indices": [0]
        }
    }

}

In [14]:
tf.keras.backend.clear_session()

lays = {
    "node_pf": NodePartitionFunctions(**lays_args["node_pf"]["kwargs"]),
    "edge_bwd": EdgeBackward(**lays_args["edge_bwd"]["kwargs"]),
    "lin_me_gnet": LinearMEGraphNet(**lays_args["lin_me_gnet"]["kwargs"])
}

def get_graph(inputs):
    nm, Tt, Ti, kf, na = inputs
    node_attr = tf.concat([nm,Tt,Ti], axis=-1)
    edge_attr = kf*na
    return (node_attr, edge_attr)

def call(inputs, training=False):
    graph = get_graph(inputs)
    # Node partition functions
    graph = lays["node_pf"](graph, training=training)
    # Edge backward
    graph = lays["edge_bwd"](graph, training=training)
    # Graph processor
    return lays["lin_me_gnet"](graph, training=training) + inputs[0]

In [15]:
graph[3]

<tf.Tensor: shape=(940, 3, 1), dtype=float64, numpy=
array([[[1.97141119e-16],
        [3.84035339e-17],
        [4.26214545e-16]],

       [[1.97141119e-16],
        [3.84035339e-17],
        [4.26214545e-16]],

       [[1.97141119e-16],
        [3.84035339e-17],
        [4.26214545e-16]],

       ...,

       [[1.97141119e-16],
        [3.84035339e-17],
        [4.26214545e-16]],

       [[1.97141119e-16],
        [3.84035339e-17],
        [4.26214545e-16]],

       [[1.97141119e-16],
        [3.84035339e-17],
        [4.26214545e-16]]])>

In [16]:
shapes = tf.nest.map_structure(lambda x: tuple(x.shape[1:]), graph)
inputs = [tf.keras.Input(shape) for shape in shapes]
model = tf.keras.Model(inputs, call(inputs))

In [17]:
def add_noise(x, stddev=0.02):
    rnd = tf.random.get_global_generator()
    dx = rnd.normal(
        shape=tf.shape(x),
        mean=0.0,
        stddev=stddev,
        dtype=x.dtype,
    )
    return x * (1 + dx)

In [20]:
graphn = tf.nest.map_structure(tf.identity, graph)
graphn = tf.nest.map_structure(lambda x: add_noise(x, stddev=0.2), graphn)
with tf.GradientTape(persistent=True) as g:
  g.watch(graphn[2])
  y = model(graphn) / const.UNA
  e = tf.reduce_mean(tf.abs(target - y)/target)
# dy_dx = g.batch_jacobian(e, graphn[1])
dy_dx = g.gradient(e, graphn[2])
dy_dx = tf.squeeze(dy_dx)
dy_dx

<tf.Tensor: shape=(940, 3), dtype=float64, numpy=
array([[-4.07760995e+02,  1.17339037e+03,  2.98524652e+00],
       [-9.55865547e+02,  3.29478168e+03,  4.11580168e+01],
       [-7.13852127e+02,  1.95758650e+03,  1.47851473e+01],
       ...,
       [-5.94821374e+03,  1.37340617e+04,  6.83610091e+01],
       [-1.48206569e+02,  6.78408053e+02,  1.63624068e+01],
       [-9.19355943e+02,  2.78281818e+03,  2.16173134e+02]])>